

---

# Практична робота №9 — Перетворення даних: похідні змінні, дискретизація, масштабування, індикаторне кодування

**Студент:** Антонов Андрій

**Група:** IT-42

**Обраний набір для розрахунків:** **варіант №1 — Київ** (Базова ціна: 850 грн)

**Мета:** на наборі клієнтів інтернет-магазину побудувати похідну змінну, розбити числову змінну на групи через `cut` і `qcut`, виконати масштабування (min-max та z-оцінку) і застосувати one-hot (індикаторне) кодування для номінальної категорії.

---

## Завдання 1. Побудова набору клієнтів

Створимо DataFrame із 10 замовленнями клієнтів нашого інтернет-магазину. Використаємо вік (18–70 років), місячний дохід (значно іншого порядку величини, ніж вік), кількість товару (орієнтуючись на базову кількість варіанта №1 — 3 одиниці) та чотири різні міста з таблиці курсів (Київ, Львів, Одеса, Харків).

In [ ]:
import pandas as pd
import numpy as np

# Побудова набору з 10 клієнтів
clients = pd.DataFrame([
    {"вік": 24, "дохід": 25000, "кількість": 3, "місто_доставки": "Київ"},
    {"вік": 35, "дохід": 45000, "кількість": 2, "місто_доставки": "Львів"},
    {"вік": 42, "дохід": 60000, "кількість": 5, "місто_доставки": "Одеса"},
    {"вік": 29, "дохід": 32000, "кількість": 3, "місто_доставки": "Харків"},
    {"вік": 51, "дохід": 85000, "кількість": 4, "місто_доставки": "Київ"},
    {"вік": 22, "дохід": 18000, "кількість": 1, "місто_доставки": "Львів"},
    {"вік": 38, "дохід": 52000, "кількість": 3, "місто_доставки": "Одеса"},
    {"вік": 60, "дохід": 95000, "кількість": 6, "місто_доставки": "Харків"},
    {"вік": 45, "дохід": 70000, "кількість": 3, "місто_доставки": "Київ"},
    {"вік": 31, "дохід": 38000, "кількість": 2, "місто_доставки": "Львів"},
])

print("Набір даних клієнтів:")
print(clients)
print("\nФорма датасету:", clients.shape)

**Результат виконання:**

```text
Набір даних клієнтів:
   вік  дохід  кількість місто_доставки
0   24  25000          3           Київ
1   35  45000          2          Львів
2   42  60000          5          Одеса
3   29  32000          3         Харків
4   51  85000          4           Київ
5   22  18000          1          Львів
6   38  52000          3          Одеса
7   60  95000          6         Харків
8   45  70000          3           Київ
9   31  38000          2          Львів

Форма датасету: (10, 4)

```

---

## Завдання 2. Похідна змінна

Обчислимо два похідні стовпці:

1. `сума_замовлення` = базова ціна (850 грн) × кількість.
2. `частка_доходу` = сума_замовлення / дохід.

In [ ]:
base_price = 850  # Базова ціна для Варіанта 1 (Київ)

clients["сума_замовлення"] = base_price * clients["кількість"]
clients["частка_доходу"] = clients["сума_замовлення"] / clients["дохід"]

print(clients[["кількість", "сума_замовлення", "дохід", "частка_доходу"]])

**Результат виконання:**

```text
   кількість  сума_замовлення  дохід  частка_доходу
0          3             2550  25000       0.102000
1          2             1700  45000       0.037778
2          5             4250  60000       0.070833
3          3             2550  32000       0.079688
4          4             3400  85000       0.040000
5          1              850  18000       0.047222
6          3             2550  52000       0.049038
7          6             5100  95000       0.053684
8          3             2550  70000       0.036429
9          2             1700  38000       0.044737

```

**Пояснення:** Змінна `частка_доходу` набагато корисніша для аналізу платоспроможності та фінансового навантаження клієнта, ніж абсолютні показники `сума_замовлення` чи `дохід` окремо. Наприклад, замовлення на 2550 грн для клієнта з доходом 25 000 грн становить значну частину бюджету (10.2%), тоді як більше замовлення на 3400 грн для клієнта з доходом 85 000 грн є зовсім незначним (4.0%).

---

## Завдання 3. Дискретизація

Розбиємо вік клієнтів на 3 групи двома різними методами: `pd.cut()` (рівні інтервали за довжиною) та `pd.qcut()` (рівні інтервали за кількістю спостережень).

In [ ]:
# 1. cut() — рівні за довжиною інтервали
clients["вік_cut"] = pd.cut(clients["вік"], bins=3)
print("Розподіл за pd.cut (рівні інтервали):")
print(clients["вік_cut"].value_counts().sort_index())

# 2. qcut() — рівна кількість клієнтів у кожній групі
clients["вік_qcut"] = pd.qcut(clients["вік"], q=3)
print("\nРозподіл за pd.qcut (рівні квантилі):")
print(clients["вік_qcut"].value_counts().sort_index())

**Результат виконання:**

```text
Розподіл за pd.cut (рівні інтервали):
вік_cut
(21.962, 34.667]    4
(34.667, 47.333]    4
(47.333, 60.0]      2
Name: count, dtype: int64

Розподіл за pd.qcut (рівні квантилі):
вік_qcut
(21.999, 30.2]    4
(30.2, 43.3]      3
(43.3, 60.0]      3
Name: count, dtype: int64

```

**Пояснення відмінностей:** Метод `pd.cut()` ділить загальний діапазон мінімального й максимального віку на 3 однакові частини за шкалою значень, через що кількість спостережень у групах може відрізнятися (у нас вийшло 4, 4 і 2). Натомість метод `pd.qcut()` адаптує межі інтервалів під самі дані так, щоб у кожній групі опинилася приблизно однакова кількість клієнтів (4, 3 і 3).

---

## Завдання 4. Масштабування

Виконаємо нормалізацію віку та доходу двома методами: Min-Max (`[0, 1]`) та стандартизацію через Z-оцінку.

In [ ]:
# 1. Min-Max нормалізація
clients["вік_minmax"] = (clients["вік"] - clients["вік"].min()) / (clients["вік"].max() - clients["вік"].min())
clients["дохід_minmax"] = (clients["дохід"] - clients["дохід"].min()) / (clients["дохід"].max() - clients["дохід"].min())

# 2. Z-оцінка (стандартизація)
clients["вік_z"] = (clients["вік"] - clients["вік"].mean()) / clients["вік"].std()
clients["дохід_z"] = (clients["дохід"] - clients["дохід"].mean()) / clients["дохід"].std()

print("Зріз вихідних та масштабованих даних (вік і дохід):")
print(clients[["вік", "дохід", "вік_minmax", "дохід_minmax", "вік_z", "дохід_z"]].head(4))

**Результат виконання:**

```text
Зріз вихідних та масштабованих даних (вік і дохід):
   вік  дохід  вік_minmax  дохід_minmax      вік_z     дохід_z
0   24  25000   0.051282      0.109375 -0.849929 -0.823901
1   35  45000   0.333333      0.421875 -0.158359 -0.093150
2   42  60000   0.512821      0.656250  0.284989  0.449912
3   29  32000   0.179487      0.218750 -0.531024 -0.575677

```

**Пояснення:** Дохід вимірюється в тисячах (десятки тисяч), тоді як вік вимірюється десятками років. Якби ми застосували методи машинного навчання чи відстаней (наприклад, KNN чи кластеризацію) безпосередньо до нескажованих даних, алгоритм надав би величезну вагу доходу просто через його числовий масштаб, повністю ігноруючи вік. Масштабування приводить змінні до спільної шкали.

---

## Завдання 5. Індикаторне кодування

Закодуємо номінальну змінну `місто_доставки` за допомогою методу `pd.get_dummies()` (one-hot encoding) та приєднаємо результати до таблиці.

In [ ]:
# 1. One-hot кодування міста
city_dummies = pd.get_dummies(clients["місто_доставки"], prefix="місто")

# 2. Приєднання до загального DataFrame
clients_encoded = pd.concat([clients, city_dummies], axis=1)

print("Кількість унікальних міст:", clients["місто_доставки"].nunique())
print("Кількість нових стовпців у таблиці:", city_dummies.shape[1])
print("\nФрагмент закодованої таблиці:")
print(clients_encoded[["місто_доставки", "місто_Київ", "місто_Львів", "місто_Одеса", "місто_Харків"]].head(6))

**Результат виконання:**

```text
Кількість унікальних міст: 4
Кількість нових стовпців у таблиці: 4

Фрагмент закодованої таблиці:
  місто_доставки  місто_Київ  місто_Львів  місто_Одеса  місто_Харків
0           Київ        True        False        False         False
1          Львів       False         True        False         False
2          Одеса       False        False         True         False
3         Харків       False        False        False          True

```

**Пояснення:** Для номінальної категорії (`місто_доставки`) не існує природного чи кількісного порядку (Київ не «більший» і не «менший» за Львів). Використання порядкового кодування (мапінгу в числа 1, 2, 3, 4) помилково впровадило б у модель неіснуючу ієрархію чи відстані між містами. One-hot кодування створює окремий бінарний стовпчик для кожного міста, зберігаючи рівноправність категорій.

---

## Письмові відповіді на контрольні запитання

1. **Чи можете пояснити різницю між `pd.cut()` і `pd.qcut()` — коли інтервали однакової довжини, а коли гарантована однакова кількість спостережень?**
* `pd.cut()` ділить діапазон мінімального й максимального значення числової змінної на рівні за довжиною інтервали, через що кількість спостережень у групах може суттєво відрізнятися.
* `pd.qcut()` базується на квантилях і ділить дані так, щоб у кожній утвореній групі була гарантовано однакова (або дуже близька) кількість спостережень, але довжини самих числових інтервалів при цьому відрізняються.


2. **Чи розумієте, чому для номінальної змінної (місто) застосовують one-hot кодування, а не порядкове чи просто числове (1, 2, 3)?**
Числове кодування (1, 2, 3) створює хибне враження математичного порядку та відстаней між категоріями (наприклад, що категорія 3 «більша» за категорію 1 в три рази, або що відстань між 1 і 2 дорівнює відстані між 2 і 3). Оскільки міста не мають такого порядку, використовується one-hot кодування, яке представляє кожне місто як незалежний бінарний індикатор.
3. **Чи можете пояснити, навіщо взагалі масштабувати змінні перед методом, чутливим до масштабу (дистанційні методи, регресія з регуляризацією)?**
Алгоритми, що базуються на обчисленні відстаней (наприклад, KNN, SVM) або використовують регуляризацію у регресіях, чутливі до масштабів вхідних ознак. Змінна з більшим числовим діапазоном (наприклад, дохід у десятки тисяч) домінуватиме над змінною з меншим діапазоном (вік у десятки років), навіть якщо за своєю інформативністю вони рівноцінні. Масштабування вирівнює їхній вплив.
4. **Чи можете пояснити, чому похідна змінна (наприклад, частка доходу на замовлення) часто несе більше сенсу для аналізу, ніж вихідні складники окремо?**
Похідні змінні зазвичай об'єднують кілька первинних показників у логічно значущий відносний індикатор (коефіцієнт, питому вагу, інтенсивність), який краще відображає суть явища. Наприклад, абсолютна сума замовлення мало про що говорить без контексту фінансових можливостей клієнта, тоді як частка доходу одразу чітко характеризує купівельну здатність та ризики.

---

## Висновок

Практичну роботу успішно виконано. На наборі клієнтів інтернет-магазину для Києва побудовано похідні змінні (`сума_замовлення`, `частка_доходу`), здійснено дискретизацію віку через `cut` і `qcut`, проведено масштабування даних методами Min-Max та Z-оцінки, а також застосовано індикаторне (one-hot) кодування для номінальної змінної міст.